# SAADE–AUA v7 — Scenario-Adaptive **Evaluation**, not Training

**Один ноутбук, без E1/E2/E3 і без модифікації ваг моделей.** Повторюємо принципи першоджерела *Scenario-Adaptive Evaluation of Trustworthy Fine-Tuned Text Models Across Knowledge-Grounded Generation and Misinformation Detection* (MAKE, 2026; DOI: 10.3390/make8060161): збалансовані сценарії → однаковий інференс → оцінювання за еталоном → інтегральний індекс → факторний аналіз / MCDM.

**Адаптація до аудіо:** модель × REAL/FAKE × тип акустичного спотворення × рівень. У статті використано *Alignment Robustness Index* (ARI) для тексту; тут окремо введено **Audio Scenario Robustness Index (ASRI-Audio)** як середнє Balanced Accuracy по 4 групах спотворень (без clean), рівні кожної групи мають однакову вагу. Не плутати з індексом чи величинами статті та попередньою ASRI гілкою навчання.

**Важливо:** у наявному v6 validation лише 1 TTS-голос (`tetiana`). Висновки про узагальнення на різні синтетичні голоси з цієї вибірки робити не можна. За замовчуванням беремо **validation**, `hidden_test` не відкривається і не оцінюється.

Джерела готових моделей: [AASIST](https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST) (ONNX, ASVspoof2019 LA), [AASIST-L](https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L) (ONNX, полегшена споріднена модель), [Wav2Vec2 (Gary Stafford)](https://huggingface.co/garystafford/wav2vec2-deepfake-voice-detector). Третю модель [Hemgg](https://huggingface.co/Hemgg/Deepfake-audio-detection) можна додати після підтвердження орієнтації класів `id2label`. **Не використовувати моделі без checkpoint або з випадковими вагами.**

## 0. Налаштування і залежності

Налаштовано шляхи сервера. `MAX_PAIRS=12` — швидка перевірка повного процесу; для основної статті встановіть `None`. Результати SMOKE та FULL зберігаються окремо. Встановлення залежностей за потреби увімкніть параметром `INSTALL_MISSING`.

In [ ]:
from pathlib import Path
import os, json, io, gc, time, math, hashlib, shutil, subprocess, sys, warnings
import tempfile, urllib.request
import numpy as np
import pandas as pd
from IPython.display import display, Audio

PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'
RESULTS_ROOT = PROJECT / 'results'
V6_DIR = RESULTS_ROOT / 'SAADE_AUA_2026_v6_2_MATCHED_TTS'
if not (V6_DIR/'V6_LOCKED_MATCHED_SPLITS.csv').exists():
    alt = RESULTS_ROOT / 'SAADE_AUA_2026_v6_MATCHED_TTS'
    if (alt/'V6_LOCKED_MATCHED_SPLITS.csv').exists(): V6_DIR = alt

MAX_PAIRS = 12                    # SMOKE; установіть None для повного дослідження
EVAL_SPLITS = ('validation',)    # НЕ додавати hidden_test сюди
RUN_INFERENCE = True
VERIFY_SHA256 = True             # перевірка фактичних WAV і маніфестів
MATCH_PAIR_DURATION = True        # однакова корисна довжина для REAL/FAKE у парі
INSTALL_MISSING = False
MODEL_ENABLED = {
    'AASIST_ONNX': True,          # ~1.6 MB, ASVspoof2019 LA
    'AASIST_L_ONNX': True,        # ~0.8 MB, легка архітектура AASIST-L
    'WAV2VEC2_STAFFORD': True,    # ~1.26 GB, потрібно більше RAM/VRAM
    'WAV2VEC2_HEMGG': False,     # тільки після явного підтвердження class mapping
}
HEMGG_FAKE_INDEX = None        # якщо id2label = LABEL_0/LABEL_1, встановлюється тільки з обґрунтуванням
CALIBRATION_PAIRS = 48          # лише split=train; поріг на clean, НЕ навчання моделей
BOOTSTRAP_REPS = 300           # CI за pair_id (не доводить перенесення на нові голоси)
SEED = 20261115
ALLOW_HIDDEN_TEST_ONCE = False # у Run All завжди False

RUN_TAG = 'FULL' if MAX_PAIRS is None else f'SMOKE_{MAX_PAIRS}'
OUT = RESULTS_ROOT / 'SAADE_AUA_2026_v7_SCENARIO_EVAL' / RUN_TAG
OUT.mkdir(parents=True,exist_ok=True)
print('DATA:',V6_DIR)
print('RESULTS:',OUT)
print('FFMPEG:',shutil.which('ffmpeg'))
assert set(EVAL_SPLITS)=={'validation'}, 'v7 публічний аналіз дозволяє тільки validation. Train використовується лише для калібрування; hidden закритий.'
if INSTALL_MISSING:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','numpy','pandas','scipy','soundfile','scikit-learn','matplotlib','transformers>=4.40,<5','torch','onnxruntime','huggingface_hub','statsmodels'])
import soundfile as sf
from scipy import signal
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve, f1_score, confusion_matrix
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['figure.figsize']=(10,5)
print('READY: Python packages imported')

## 1. Маніфест REAL–TTS та контроль незалежності

Використовуємо **вже синтезовані WAV із v6**, не генеруємо заново. Перевіряємо 1 REAL + 1 FAKE на `pair_id`, шляхи, доступність, групи людей і голоси TTS. Усі файли AUA — тільки для читання.

In [ ]:
MANIFEST_PATH = V6_DIR / 'V6_LOCKED_MATCHED_SPLITS.csv'
assert MANIFEST_PATH.is_file(), f'Немає {MANIFEST_PATH}. Спочатку потрібні готові REAL–TTS WAV із v6.'
raw = pd.read_csv(MANIFEST_PATH, low_memory=False)
needed = {'path','label','pair_id','split','tts_voice_id','group_id','parent_file_id'}
assert needed.issubset(raw.columns), f'Відсутні колонки: {needed-set(raw.columns)}'
raw['label'] = pd.to_numeric(raw.label,errors='raise').astype(int)
assert raw.label.isin([0,1]).all()
assert raw['pair_id'].notna().all() and raw['path'].notna().all()
# Захист від прихованої вибірки на рівні аналізу, кешів та прослуховування
raw_public = raw.loc[~raw.split.eq('hidden_test')].copy()
manifest = raw_public.loc[raw_public.split.isin(EVAL_SPLITS)].copy()
counts = manifest.groupby(['pair_id','label']).size().unstack(fill_value=0)
assert (counts.get(0,pd.Series(dtype=int)).eq(1) & counts.get(1,pd.Series(dtype=int)).eq(1)).all(), 'Порушено відповідність REAL–FAKE 1:1'
assert manifest.groupby('pair_id').split.nunique().max()==1
print('Усього в маніфесті:',len(raw),'рядків; відкрито validation:',len(manifest))
print('Розподіл класів:',manifest.label.value_counts().to_dict())
print('Голоси FAKE за розділами:')
display(pd.crosstab(raw_public.loc[raw_public.label.eq(1),'tts_voice_id'],raw_public.loc[raw_public.label.eq(1),'split']))

if MAX_PAIRS is not None:
    # Беремо всі записи відібраних пар без порушення 50/50.
    ids = sorted(manifest.pair_id.unique())
    rng = np.random.default_rng(SEED)
    ids = sorted(rng.choice(ids,size=min(MAX_PAIRS,len(ids)),replace=False))
    manifest = manifest[manifest.pair_id.isin(ids)].copy()

# Перевірка фактичних файлів — без підміни метаданих або склеювання з архіву ZIP
missing = [p for p in manifest.path.unique() if not Path(p).is_file()]
if missing:
    print('Приклад відсутнього WAV:',missing[:5])
    raise FileNotFoundError(f'{len(missing)} аудіофайлів не знайдено; перевірте V6_DIR і шляхи. Нічого не копіюйте в hidden test.')
assert len(manifest)==2*manifest.pair_id.nunique()
if VERIFY_SHA256:
    receipts_path=V6_DIR/'V6_TTS_RECEIPTS.csv'
    if not receipts_path.exists():raise FileNotFoundError('Потрібен V6_TTS_RECEIPTS.csv для перевірки TTS WAV')
    receipts=pd.read_csv(receipts_path)
    expected_tts={(str(r.pair_id),str(r.generated_path)):str(r.sha256).lower() for r in receipts.itertuples(index=False)}
    qa=[]
    for r in manifest.itertuples(index=False):
        expected = None
        if int(r.label)==1:
            expected = expected_tts.get((str(r.pair_id),str(r.path)))
            if not expected: raise RuntimeError(f'TTS немає в receipts: {r.pair_id} {r.path}')
        elif isinstance(getattr(r,'working_sha256',None),str):
            expected=str(r.working_sha256).lower()
        if expected and len(expected)==64:
            actual=hashlib.sha256(Path(r.path).read_bytes()).hexdigest()
            if actual!=expected:raise RuntimeError(f'Контрольна сума WAV не збігається: {r.path}')
            qa.append(dict(pair_id=r.pair_id,label=int(r.label),sha256_verified=True))
    pd.DataFrame(qa).to_csv(OUT/'01_HASH_AUDIT.csv',index=False)
print('Перевірено пар:',manifest.pair_id.nunique())

# Аудит перетинів між усіма split без читання звуку прихованого тесту
by_pair = raw.groupby('pair_id').split.nunique()
if (by_pair>1).any(): raise RuntimeError('PAIR LEAKAGE між split')
human_real = raw[raw.label.eq(0)].dropna(subset=['group_id'])
by_group = human_real.groupby('group_id').split.nunique()
if (by_group>1).any(): raise RuntimeError('REAL SPEAKER LEAKAGE між split')
voices = raw[raw.label.eq(1)].groupby('split').tts_voice_id.unique().to_dict()
quality = {'pairs_available_validation':int(raw.loc[raw.split.eq('validation'),'pair_id'].nunique()),
           'pairs_selected':int(manifest.pair_id.nunique()),
           'synthetic_voice_ids_validation':sorted(raw.loc[(raw.split.eq('validation')) & raw.label.eq(1),'tts_voice_id'].dropna().unique().tolist()),
           'different_tts_engines_known':1, 'real_speaker_leak':False,'pair_leak':False,
           'conclusion_scope':'within validation voice only; cannot claim cross-voice or cross-generator generalization'}
(OUT/'01_DATA_READINESS.json').write_text(json.dumps(quality,ensure_ascii=False,indent=2),encoding='utf-8')
manifest.to_csv(OUT/'01_VALIDATION_MANIFEST.csv',index=False)
print('Збережено',OUT/'01_VALIDATION_MANIFEST.csv')

## 2. Рівно 21 акустичний сценарій

1 чистий запис; 5 рівнів × 4 категорії. MP3 — **справжнє кодування/декодування FFmpeg**, а не інтерпретація рівня «як MP3». Якщо FFmpeg відсутній, код зупиниться: неповне покриття не можна назвати 21 сценарієм.

In [ ]:
assert shutil.which('ffmpeg'), 'Потрібен ffmpeg для реальних MP3 перетворень. Встановіть його на сервері.'
SR=16000
WINDOW_SEC=4.0
WINDOW_N=int(SR*WINDOW_SEC)
SCENARIOS=[{'scenario':'clean','family':'clean','level':0,'parameter':None}]
for level,snr in enumerate([30,25,20,15,10],1): SCENARIOS.append(dict(scenario=f'noise_snr{snr}',family='noise',level=level,parameter=snr))
for level,kbps in enumerate([128,96,64,48,32],1): SCENARIOS.append(dict(scenario=f'mp3_{kbps}kbps',family='mp3',level=level,parameter=kbps))
for level,rt in enumerate([.15,.30,.45,.65,.85],1): SCENARIOS.append(dict(scenario=f'reverb_rt{rt:.2f}',family='reverb',level=level,parameter=rt))
for level,hz in enumerate([7500,6000,4500,3000,2000],1): SCENARIOS.append(dict(scenario=f'lowpass_{hz}Hz',family='lowpass',level=level,parameter=hz))
SC=pd.DataFrame(SCENARIOS)
assert len(SC)==21 and SC.scenario.is_unique
SC.to_csv(OUT/'02_SCENARIO_PROTOCOL.csv',index=False)
display(SC)
# Контроль тривалості: обидва класи однієї пари отримують однаковий обрізаний інтервал.
# Це частково зменшує confound через різну тривалість REAL і TTS.
pair_limits={}
if MATCH_PAIR_DURATION:
    ids_for_duration=set(manifest.pair_id.astype(str)) | set(raw_public.loc[raw_public.split.eq('train'),'pair_id'].astype(str))
    subset=raw_public[raw_public.pair_id.astype(str).isin(ids_for_duration)]
    for pid,g in subset.groupby('pair_id'):
        if set(g.label)!={0,1}:continue
        durations=[]
        for row in g.itertuples(index=False):
            path=Path(row.path)
            if path.is_file():
                dur=sf.info(path).duration - float(getattr(row,'segment_start_sec',0) or 0)
                if dur>0:durations.append(dur)
        if len(durations)==2:
            pair_limits[str(pid)]=int(np.clip(math.floor(SR*min(durations)),1,WINDOW_N))
    if not set(manifest.pair_id.astype(str)).issubset(pair_limits):
        raise RuntimeError('Не вдалося підтвердити тривалість обох записів усіх validation пар')
    pd.DataFrame([dict(pair_id=k,effective_duration_sec=v/SR) for k,v in pair_limits.items()]).to_csv(OUT/'02_PAIR_DURATION_POLICY.csv',index=False)
    print('Спільний інтервал REAL–FAKE зафіксовано для',len(pair_limits),'пар')

def stable_seed(*fields):
    b='||'.join(map(str,fields)).encode()
    return int.from_bytes(hashlib.sha256(b).digest()[:8],'big')%(2**32)

def read_wave(path,start_sec=0,pair_id=None):
    with sf.SoundFile(str(path)) as f:
        sr=f.samplerate
        f.seek(max(0,int(round(float(start_sec)*sr))))
        wave=f.read(frames=int(math.ceil(WINDOW_SEC*sr)),dtype='float32',always_2d=True).mean(axis=1)
    if sr!=SR:
        from fractions import Fraction
        fr=Fraction(SR,sr)
        wave=signal.resample_poly(wave,fr.numerator,fr.denominator)
    wave=np.asarray(wave,dtype=np.float32)
    if pair_id is not None and MATCH_PAIR_DURATION:
        wave=wave[:pair_limits[str(pair_id)]]
    if len(wave)==0: raise ValueError(f'Порожнє аудіо: {path}')
    if len(wave)<WINDOW_N: wave=np.pad(wave,(0,WINDOW_N-len(wave)))
    return np.nan_to_num(wave[:WINDOW_N],nan=0.,posinf=0.,neginf=0.).astype(np.float32)

def mp3_roundtrip(x,kbps):
    with tempfile.TemporaryDirectory() as t:
        wd=Path(t); orig=wd/'orig.wav'; enc=wd/'encoded.mp3'; dec=wd/'decoded.wav'
        sf.write(orig,x,SR,subtype='PCM_16')
        en=['ffmpeg','-nostdin','-y','-hide_banner','-loglevel','error','-i',str(orig),'-codec:a','libmp3lame','-b:a',str(kbps)+'k','-ar',str(SR),'-ac','1',str(enc)]
        de=['ffmpeg','-nostdin','-y','-hide_banner','-loglevel','error','-i',str(enc),'-ar',str(SR),'-ac','1',str(dec)]
        subprocess.run(en,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
        subprocess.run(de,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
        with sf.SoundFile(dec) as f: y=f.read(dtype='float32',always_2d=True).mean(axis=1)
    return np.pad(y[:len(x)],(0,max(0,len(x)-len(y)))).astype(np.float32)

def apply_scenario(x, row, sample_key):
    family=row['family']; val=row['parameter']
    if family=='clean': return x.copy()
    if family=='noise':
        rng=np.random.default_rng(stable_seed(sample_key,row['scenario']))
        n=rng.standard_normal(len(x)).astype(np.float32)
        sig=np.sqrt(float(np.mean(x.astype(np.float64)**2))+1e-12)
        nr=np.sqrt(float(np.mean(n.astype(np.float64)**2))+1e-12)
        return np.clip(x+(sig/(10**(float(val)/20)))/nr*n,-1,1).astype(np.float32)
    if family=='mp3': return mp3_roundtrip(x,int(val))
    if family=='reverb':
        # Синтетичний IR, керований RT60, НЕ виміряне реальне приміщення
        rt=float(val); nr=int(min(SR*1.5,SR*rt*1.2))
        t=np.arange(nr)/SR; envelope=np.exp(-6.9078*t/rt)
        rng=np.random.default_rng(stable_seed('IR',row['scenario']))
        h=(rng.normal(size=nr).astype(np.float32)*envelope).astype(np.float32)
        h[0]+=4.0
        h=h/np.sqrt(np.sum(h.astype(np.float64)**2))
        y=signal.fftconvolve(x,h,mode='full')[:len(x)]
        # однаковий gain для кожної конфігурації, без per-clip peak normalization
        return np.clip(y,-1,1).astype(np.float32)
    if family=='lowpass':
        sos=signal.butter(6,float(val),btype='lowpass',fs=SR,output='sos')
        return signal.sosfilt(sos,x).astype(np.float32)
    raise KeyError(family)

# Тест протоколу без доступу до моделей, 21 тип і відтворюваність шуму
probe=(0.25*np.sin(2*np.pi*440*np.arange(WINDOW_N)/SR)).astype(np.float32)
assert all(apply_scenario(probe,r,'test').shape==(WINDOW_N,) for r in SCENARIOS)
assert np.array_equal(apply_scenario(probe,SCENARIOS[1],'same'),apply_scenario(probe,SCENARIOS[1],'same'))
print('21/21 сценарій працює; noise відтворюваний, FFmpeg MP3 перевірено')

## 3. Фіксовані моделі без навчання

Моделі завантажуються тільки з готових checkpoint. Перед оцінюванням `model.eval()` і `torch.inference_mode()`. Усі моделі отримують **однаковий** 16-kHz моно сигнал тривалістю 4 с. Для AASIST ONNX — перші 64600 семплів, як в опублікованому протоколі. Для Wav2Vec2 використовується опублікована `AutoFeatureExtractor` та класова орієнтація із `id2label`. Якщо модель має лише `LABEL_0/LABEL_1`, орієнтацію необхідно вказати й задокументувати явно.

In [ ]:
MODEL_SOURCES={
 'AASIST_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST','file':'aasist.onnx','higher_is_bonafide':True,
                 'source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST'},
 'AASIST_L_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST-L','file':'aasist-l.onnx','higher_is_bonafide':True,
                 'source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L'},
 'WAV2VEC2_STAFFORD':{'kind':'hf','repo':'garystafford/wav2vec2-deepfake-voice-detector','expected_fake_index':1,
                    'source':'https://huggingface.co/garystafford/wav2vec2-deepfake-voice-detector'},
 'WAV2VEC2_HEMGG':{'kind':'hf','repo':'Hemgg/Deepfake-audio-detection','expected_fake_index':HEMGG_FAKE_INDEX,
                   'source':'https://huggingface.co/Hemgg/Deepfake-audio-detection'},
}
MODELS_DIR=RESULTS_ROOT/'SAADE_AUA_2026_v7_SCENARIO_EVAL'/'pretrained_model_cache'
MODELS_DIR.mkdir(parents=True,exist_ok=True)

def download_file_identity(repo, filename, destination, min_bytes=50000):
    destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.is_file() and destination.stat().st_size>=min_bytes:
        return destination
    url=f'https://huggingface.co/{repo}/resolve/main/{filename}'
    req=urllib.request.Request(url,headers={'User-Agent':'SAADE-eval/7.0','Accept-Encoding':'identity'})
    tmp=destination.with_name(destination.name+'.part')
    try:
        with urllib.request.urlopen(req,timeout=180) as resp, open(tmp,'wb') as out:
            while True:
                b=resp.read(1024*1024)
                if not b:break
                out.write(b)
        if tmp.stat().st_size<min_bytes: raise ValueError('Remote file too small; possible HTML/API error: '+filename)
        if tmp.open('rb').read(32).lstrip().lower().startswith(b'<html'):
            raise ValueError('Expected a model file, got HTML response: '+filename)
        os.replace(tmp,destination)
    finally:
        tmp.unlink(missing_ok=True)
    return destination

def get_fake_index(id2label,expected=None):
    labels={int(i):str(v).lower().strip() for i,v in id2label.items()}
    fake=[i for i,v in labels.items() if any(w in v for w in ('fake','spoof','synthetic','generated','deepfake'))
          and not any(w in v for w in ('real','bonafide','bona fide','genuine'))]
    if len(fake)==1:idx=fake[0]
    elif expected is not None:idx=int(expected)
    else:raise ValueError(f'Невідома орієнтація класів {labels}. Вкажіть і обґрунтуйте fake index.')
    if idx not in labels or len(labels)!=2:raise ValueError(f'Очікуємо 2 класи, отримано {labels}')
    if expected is not None and int(expected)!=idx:raise ValueError(f'Невідповідність label mapping: {labels}')
    return idx,labels

class HFDetector:
    def __init__(self,name,config):
        import torch
        from transformers import AutoFeatureExtractor,AutoModelForAudioClassification
        self.name=name;self.cfg=config; self.torch=torch
        self.device='cuda' if torch.cuda.is_available() else 'cpu'
        # Hugging Face може падати через несумісний httpx2/Brotli на Jupyter-сервері.
        # У такому випадку переносимо лише потрібні model files через urllib identity.
        repo=config['repo']
        try:
            self.fe=AutoFeatureExtractor.from_pretrained(repo,trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(repo,trust_remote_code=False)
        except Exception as download_exc:
            print('Стандартний HF loader не спрацював:',repr(download_exc)[:300])
            print('Fallback: HTTPS identity, без Brotli; pretrained model files only')
            cache=MODELS_DIR/name
            for fn in ['config.json','preprocessor_config.json','model.safetensors']:
                download_file_identity(repo,fn,cache/fn,min_bytes=(50000 if fn.endswith('.safetensors') else 100))
            self.fe=AutoFeatureExtractor.from_pretrained(str(cache),local_files_only=True,trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(str(cache),local_files_only=True,trust_remote_code=False)
        self.fake_idx,self.labels=get_fake_index(self.model.config.id2label,config.get('expected_fake_index'))
        self.model=self.model.to(self.device).eval()
        self.model.requires_grad_(False)
        self.model_revision=getattr(self.model.config,'_commit_hash',None) or 'unknown'
        self.score_kind='fake_probability'
    def score_fake(self,wave):
        x=self.fe(wave,sampling_rate=SR,return_tensors='pt',padding=True)
        x={k:v.to(self.device) for k,v in x.items()}
        with self.torch.inference_mode():
            z=self.model(**x).logits
            p=self.torch.softmax(z,dim=-1)[0,self.fake_idx].item()
        return float(p)
    def close(self):
        del self.model
        if self.device=='cuda':self.torch.cuda.empty_cache()

class AASISTONNXDetector:
    def __init__(self,name,config):
        import onnxruntime as ort
        self.name=name;self.cfg=config
        fp=download_file_identity(config['repo'],config['file'],MODELS_DIR/config['file'])
        self.weights_sha256=hashlib.sha256(fp.read_bytes()).hexdigest()
        self.sess=ort.InferenceSession(str(fp),providers=['CPUExecutionProvider'])
        ins=self.sess.get_inputs()
        if len(ins)!=1:raise ValueError(f'Непідтримуваний ONNX input shape: {[(i.name,i.shape) for i in ins]}')
        self.input_name=ins[0].name
        self.input_shape=ins[0].shape
        self.score_kind='negative_bonafide_logit'
        self.model_revision=self.weights_sha256
    def score_fake(self,wave):
        x=np.asarray(wave[:64600],np.float32)
        if len(x)<64600:x=np.pad(x,(0,64600-len(x)))
        shape=self.input_shape
        if len(shape)==2:inp=x[None,:]
        elif len(shape)==3 and shape[1] in (1,'1',None):inp=x[None,None,:]
        else:raise ValueError(f'Невідомий input shape AASIST: {shape}')
        outs=self.sess.run(None,{self.input_name:inp})
        val=np.asarray(outs[0],dtype=np.float64).reshape(-1)
        # У clovaai/aasist/main.py score = batch_out[:,1] (bona fide).
        # ONNX wrapper може повертати вже один scalar або два logits.
        if val.size==1: bonafide_score=float(val[0])
        elif val.size==2: bonafide_score=float(val[1])
        else:
            raise ValueError('Неоднозначна структура виходу AASIST ONNX: '+str([np.asarray(z).shape for z in outs]))
        # Монотонна шкала, БІЛЬШЕ = FAKE. Це не ймовірність.
        return -bonafide_score
    def close(self): del self.sess

def load_detector(name):
    cfg=MODEL_SOURCES[name]
    if cfg['kind']=='onnx':return AASISTONNXDetector(name,cfg)
    return HFDetector(name,cfg)

print('Активні моделі:',[n for n,on in MODEL_ENABLED.items() if on])
print('Checkpoint URL та орієнтація score збережуться у 03_MODEL_PROVENANCE.json')

## 4. Фіксація порога **без навчання**

Оскільки вихід AASIST — logit, а Wav2Vec2 — ймовірність, поріг слід встановити **на окремих train-записах** і потім не змінювати на validation. Це **калібрування рішення**, а не навчання ваг. Для наукового звіту зберігаємо поріг, алгоритм і кількість пар. Окремо аналізуємо ROC-AUC/EER, що не потребують порога.

In [ ]:
def get_train_calibration():
    t=raw_public.loc[raw_public.split.eq('train')].copy()
    if t.empty:raise ValueError('Немає train для незалежного вибору порога')
    ids=sorted(t.pair_id.unique())
    rng=np.random.default_rng(SEED+1)
    chosen=sorted(rng.choice(ids,size=min(CALIBRATION_PAIRS,len(ids)),replace=False))
    t=t[t.pair_id.isin(chosen)].copy()
    assert t.label.value_counts().to_dict()=={0:len(chosen),1:len(chosen)}
    assert not set(chosen)&set(manifest.pair_id)
    for p in t.path:
        if not Path(p).is_file():raise FileNotFoundError(p)
    return t

def best_ba_threshold(y,score):
    y=np.asarray(y).astype(int); score=np.asarray(score,dtype=float)
    if len(np.unique(y))!=2 or not np.isfinite(score).all():raise ValueError('Немає двох класів або NaN')
    unique=np.unique(score)
    if len(unique)==1:return float(unique[0]),float(.5)
    thresholds=(unique[:-1]+unique[1:])/2
    thresholds=np.concatenate([[np.nextafter(unique[0],-np.inf)],thresholds,[np.nextafter(unique[-1],np.inf)]])
    vals=np.array([balanced_accuracy_score(y,(score>=t).astype(int)) for t in thresholds])
    best=np.flatnonzero(np.isclose(vals,vals.max()))
    return float(thresholds[best[len(best)//2]]),float(vals.max())

calibration=get_train_calibration()
calib_meta={}
print('Калібрування порога на clean train:',calibration.pair_id.nunique(),'пар')

## 5. Інференс для кожного сценарію (resumable)

Для кожної пари, двох класів і 21 сценарію виконуємо **той самий інференс**. Прогнози записуються інкрементально у CSV, щоб після збою не повторювати вже виконане. Ключ включає модель, сценарій, шлях, схему протоколу і версію моделі. Немає градієнтів або оптимізатора.

In [ ]:
PROTOCOL_ID='v7-eval-4sec-16k-noise-mp3-reverb-lowpass-20261115'
PREDICTIONS=OUT/'05_VALIDATION_PREDICTIONS.csv'
METADATA=OUT/'03_MODEL_PROVENANCE.json'
selected_models=[name for name,on in MODEL_ENABLED.items() if on]
if not selected_models:raise ValueError('Оберіть принаймні одну готову модель')

# Запобігання змішуванню оцінок із різних конфігурацій
config_file=OUT/'00_RUN_CONFIG.json'
run_settings=dict(protocol=PROTOCOL_ID,max_pairs=MAX_PAIRS,seed=SEED,eval_splits=EVAL_SPLITS,
                  models=selected_models,calibration_pairs=CALIBRATION_PAIRS,match_pair_duration=MATCH_PAIR_DURATION,
                  manifest_hash=hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
                  selected_pair_ids_sha256=hashlib.sha256('|'.join(sorted(manifest.pair_id.unique())).encode()).hexdigest())
if config_file.exists():
    before=json.loads(config_file.read_text())
    if before!=json.loads(json.dumps(run_settings)):
        raise RuntimeError('У цій папці результати іншого експерименту. Змініть RUN_TAG / OUTPUT замість змішування CSV.')
else:config_file.write_text(json.dumps(run_settings,ensure_ascii=False,indent=2))

if PREDICTIONS.exists():
    predictions=pd.read_csv(PREDICTIONS)
    assert 'protocol_id' in predictions and predictions.protocol_id.eq(PROTOCOL_ID).all()
else:
    predictions=pd.DataFrame()
model_info=json.loads(METADATA.read_text()) if METADATA.exists() else {}

def update_csv(new_rows):
    global predictions
    if not new_rows:return
    df=pd.DataFrame(new_rows)
    if predictions.empty:predictions=df.copy()
    else:predictions=pd.concat([predictions,df],ignore_index=True)
    predictions=predictions.drop_duplicates(subset=['model','scenario','pair_id','label','protocol_id','model_revision'],keep='last')
    temp=PREDICTIONS.with_suffix('.tmp')
    predictions.to_csv(temp,index=False)
    os.replace(temp,PREDICTIONS)

def evaluate_one_model(name):
    global predictions
    print('\nМодель:',name)
    detector=load_detector(name)
    try:
        model_info[name]={'source':MODEL_SOURCES[name]['source'], 'model_revision':detector.model_revision,
                          'score_kind':detector.score_kind,'id2label':getattr(detector,'labels',None),
                          'weights_sha256':getattr(detector,'weights_sha256',None),'inference_only':True}
        METADATA.write_text(json.dumps(model_info,ensure_ascii=False,indent=2))
        cdf=calibration.copy();cs=[]
        for r in cdf.itertuples(index=False):
            wave=read_wave(r.path,getattr(r,'segment_start_sec',0),r.pair_id)
            cs.append(detector.score_fake(wave))
        threshold,ba=best_ba_threshold(cdf.label,cs)
        calib_meta[name]={'threshold':threshold,'calibration_clean_BA':ba,'calibration_pairs':int(cdf.pair_id.nunique()),
                          'model_revision':detector.model_revision,'calibration_split':'train'}
        (OUT/'04_CALIBRATION.json').write_text(json.dumps(calib_meta,ensure_ascii=False,indent=2))
        print('Поріг:',round(threshold,6),'train BA:',round(ba,4))
        seen=set()
        if not predictions.empty:
            p=predictions.loc[(predictions.model==name)&(predictions.model_revision==detector.model_revision)]
            seen={(str(r.scenario),str(r.pair_id),int(r.label)) for r in p.itertuples(index=False)}
        pending=[];t0=time.time()
        for r in manifest.itertuples(index=False):
            scenarios=[s for s in SCENARIOS if (s['scenario'],str(r.pair_id),int(r.label)) not in seen]
            if not scenarios:continue
            wave=read_wave(r.path,getattr(r,'segment_start_sec',0),r.pair_id)
            for s in scenarios:
                transformed=apply_scenario(wave,s,f'{r.pair_id}/{r.label}')
                score=detector.score_fake(transformed)
                if not np.isfinite(score):raise ValueError(f'NaN score {name} {r.pair_id} {s["scenario"]}')
                pending.append(dict(protocol_id=PROTOCOL_ID,model=name,model_revision=detector.model_revision,
                                    scenario=s['scenario'],family=s['family'],level=s['level'],parameter=s['parameter'],
                                    pair_id=r.pair_id,label=int(r.label),path=str(r.path),
                                    speaker_group=str(r.group_id),tts_voice_id=str(r.tts_voice_id),
                                    score_fake=score,threshold=threshold,pred_label=int(score>=threshold)))
            if len(pending)>=42: update_csv(pending);pending=[]
        update_csv(pending)
        print('Закінчено. Час, с:',round(time.time()-t0,1))
    finally:
        detector.close(); del detector;gc.collect()

if RUN_INFERENCE:
    for modelname in selected_models:
        evaluate_one_model(modelname)
else:
    print('RUN_INFERENCE=False: лише аналіз існуючого CSV')
assert not predictions.empty,'Прогнози відсутні — запустіть готові моделі'
# Якщо відновили з CSV, пороги беруться саме із CSV попередніх запусків
for name in selected_models:
    m=predictions[predictions.model==name]
    expected=2*len(manifest.pair_id.unique())*len(SCENARIOS)
    assert len(m)==expected, f'{name}: {len(m)}/{expected} передбачень; результати неповні'
    assert m.threshold.nunique()==1, f'{name}: різні пороги. Не можна змішувати запуски'
print('Загальна кількість прогнозів:',len(predictions))

## 6. Метрики, ASRI-Audio, падіння відносно clean

**ASRI-Audio** = середнє *Balanced Accuracy* за чотирма категоріями, усереднюючи 5 рівнів усередині категорії. Категорії мають рівну вагу незалежно від кількості фрагментів. Не обчислюємо класову точність із одного класу й не підміняємо відсутній ROC-AUC нулем. EER у шкалі 0–1: нижче краще.

In [ ]:
def eer_from_scores(y,s):
    if len(np.unique(y))<2:return np.nan
    fpr,tpr,thr=roc_curve(y,s)
    fnr=1-tpr
    k=int(np.argmin(np.abs(fpr-fnr)))
    if k==0:return float((fpr[0]+fnr[0])/2)
    # Апроксимація точки перетину EER між сусідніми точками
    cross=np.where(np.sign((fpr-fnr)[1:])!=np.sign((fpr-fnr)[:-1]))[0]
    if len(cross):
        i=int(cross[0]);a=(fpr-fnr)[i];b=(fpr-fnr)[i+1]
        w=0 if b==a else -a/(b-a)
        return float(fpr[i]+w*(fpr[i+1]-fpr[i]))
    return float((fpr[k]+fnr[k])/2)

def evaluate_metrics(df):
    y=df.label.to_numpy().astype(int); scores=df.score_fake.to_numpy(float)
    pred=df.pred_label.to_numpy().astype(int)
    cm=confusion_matrix(y,pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    return dict(n=len(df),pairs=df.pair_id.nunique(), n_real=int((y==0).sum()),n_fake=int((y==1).sum()),
                BA=float(balanced_accuracy_score(y,pred)) if len(np.unique(y))==2 else np.nan,
                ROC_AUC=float(roc_auc_score(y,scores)) if len(np.unique(y))==2 else np.nan,
                EER=float(eer_from_scores(y,scores)),F1_fake=float(f1_score(y,pred,zero_division=0)),
                TPR_fake=float(tp/max(tp+fn,1)),TNR_real=float(tn/max(tn+fp,1)),
                TP=int(tp),TN=int(tn),FP=int(fp),FN=int(fn))

metrics=[]
for (model,scenario,family,level),g in predictions.groupby(['model','scenario','family','level'],dropna=False,sort=True):
    metrics.append({'model':model,'scenario':scenario,'family':family,'level':level,**evaluate_metrics(g)})
metrics=pd.DataFrame(metrics)
metrics.to_csv(OUT/'06_SCENARIO_METRICS.csv',index=False)
# Category scores: mean of *scenario BA* rather than pooling unequal category sizes
category=metrics[metrics.family!='clean'].groupby(['model','family'],as_index=False).agg(BA=('BA','mean'),ROC_AUC=('ROC_AUC','mean'),EER=('EER','mean'))
category.to_csv(OUT/'06_CATEGORY_METRICS.csv',index=False)
rows=[]
for m,g in metrics.groupby('model'):
    stress=g[g.family!='clean']; clean=g[g.family=='clean'].iloc[0]
    cat=category[category.model==m]
    assert len(cat)==4 and len(stress)==20
    asri=float(cat.BA.mean())
    rows.append(dict(model=m,ASRI_Audio=asri,clean_BA=float(clean.BA),drop_clean_to_ASRI=float(clean.BA-asri),
                     worst_scenario_BA=float(stress.BA.min()),mean_scenario_AUC=float(stress.ROC_AUC.mean()),
                     mean_scenario_EER=float(stress.EER.mean()),clean_ROC_AUC=float(clean.ROC_AUC),
                     n_pairs=int(clean.pairs)))
summary=pd.DataFrame(rows).sort_values('ASRI_Audio',ascending=False)
summary.to_csv(OUT/'06_ASRI_AUDIO_SUMMARY.csv',index=False)
display(summary)
print('Примітка: цей ASRI-Audio має шкалу 0..1 і не є ARI зі статті MAKE')
# Рейтинг на clean vs. рейтинг на кожному зі сценаріїв; не обираємо переможця при нічиїй
def unique_winner(df):
    max_ba=df.BA.max()
    winners=df.loc[np.isclose(df.BA,max_ba),'model'].tolist()
    return winners[0] if len(winners)==1 else 'TIE'
clean_winner=unique_winner(metrics[metrics.family=='clean'])
reversals=[]
for scenario,g in metrics.groupby('scenario'):
    if scenario=='clean':continue
    winner=unique_winner(g)
    reversals.append(dict(scenario=scenario,clean_best_model=clean_winner,scenario_best_model=winner,
                          rank_reversal=bool(winner!='TIE' and clean_winner!='TIE' and winner!=clean_winner)))
pd.DataFrame(reversals).to_csv(OUT/'06_RANK_REVERSALS.csv',index=False)
print('Зміна найкращої моделі відносно clean у сценаріях:',sum(z['rank_reversal'] for z in reversals),'з',len(reversals))

## 7. Перевірка надійності: парний bootstrap, факторний аналіз

Bootstrap виконуємо за **парами REAL–TTS**, а не за тисячами корельованих сценарних передбачень. Це дає умовний інтервал **для наявних голосів**, не переноситься на нові TTS-рушії. Факторна модель із кластер-стандартними похибками за `pair_id` — описова оцінка взаємодії «детектор × категорія», не заміна повноцінної генераторно-незалежної валідації.

In [ ]:
# Стратифікований за pair_id bootstrap для ASRI і парної різниці між моделями
rng=np.random.default_rng(SEED+7)
all_pairs=np.array(sorted(predictions.pair_id.unique()))
models=sorted(predictions.model.unique())
# Попередньо створені масиви true/score/pred по model,scenario,pair,label
samples={}
for (m,s),g in predictions.groupby(['model','scenario']):
    q=g.pivot(index='pair_id',columns='label',values='pred_label').reindex(all_pairs)
    assert q.notna().all().all(),f'Неповні пари {m},{s}'
    samples[(m,s)]=q[[0,1]].to_numpy(dtype=int)
def fast_ba(labels2):
    # labels2 shape (n_pairs,2), col0=real col1=fake
    return float(.5*(np.mean(labels2[:,0]==0)+np.mean(labels2[:,1]==1)))
def bootstrap_asri(m,ids):
    cat_values=[]
    for family in ('noise','mp3','reverb','lowpass'):
        vals=[fast_ba(samples[(m,s['scenario'])][ids]) for s in SCENARIOS if s['family']==family]
        cat_values.append(np.mean(vals))
    return float(np.mean(cat_values))
ci=[]; contrasts=[]
rep_values={m:[] for m in models}
for _ in range(BOOTSTRAP_REPS):
    ix=rng.integers(0,len(all_pairs),size=len(all_pairs))
    for m in models:rep_values[m].append(bootstrap_asri(m,ix))
for m in models:
    v=np.asarray(rep_values[m]);ci.append(dict(model=m,estimate=float(summary.set_index('model').loc[m,'ASRI_Audio']),
                        ci_low=float(np.quantile(v,.025)),ci_high=float(np.quantile(v,.975)),
                        cluster_unit='pair_id',not_independent_voice_level=True))
for i,m1 in enumerate(models):
    for m2 in models[i+1:]:
        diff=np.asarray(rep_values[m1])-np.asarray(rep_values[m2])
        contrasts.append(dict(model_A=m1,model_B=m2,estimate=float(summary.set_index('model').loc[m1,'ASRI_Audio']-summary.set_index('model').loc[m2,'ASRI_Audio']),
                              ci_low=float(np.quantile(diff,.025)),ci_high=float(np.quantile(diff,.975))))
ci=pd.DataFrame(ci);ci.to_csv(OUT/'07_PAIR_BOOTSTRAP_CI.csv',index=False)
pd.DataFrame(contrasts).to_csv(OUT/'07_PAIRED_MODEL_CONTRASTS.csv',index=False)
display(ci)

try:
    from statsmodels.stats.anova import AnovaRM
    df_rm=predictions.copy()
    df_rm['correct']=(df_rm.label==df_rm.pred_label).astype(float)
    df_rm=(df_rm.groupby(['pair_id','model','family'],as_index=False)['correct'].mean())
    if df_rm.model.nunique()>1 and df_rm.family.nunique()>1 and df_rm.pair_id.nunique()>3:
        rm=AnovaRM(df_rm,depvar='correct',subject='pair_id',within=['model','family']).fit()
        rm.anova_table.to_csv(OUT/'07_REPEATED_MEASURES_ANOVA.csv')
        print('Repeated-measures ANOVA (pair as subject):')
        display(rm.anova_table)
    else:print('Repeated-measures ANOVA: недостатньо пар/моделей у SMOKE')
except Exception as exc:
    (OUT/'07_ANOVA_STATUS.txt').write_text('Не обчислено: '+repr(exc),encoding='utf-8')
    print('Repeated-measures ANOVA не виконано:',str(exc)[:200])

try:
    import statsmodels.formula.api as smf
    small=predictions.copy()
    small['correct']=(small.label==small.pred_label).astype(int)
    # кластер-робастна лінійна модель імовірності; головне — величина впливу та довірчі інтервали
    result=smf.ols('correct ~ C(model)*C(family) + C(label)',data=small).fit(cov_type='cluster',cov_kwds={'groups':small.pair_id})
    (OUT/'07_CLUSTER_ROBUST_FACTORIAL.txt').write_text(result.summary().as_text()+
      '\nCAUTION: clustered linear probability model, not independent-observation ANOVA; generalization limited to sampled voices.\n',encoding='utf-8')
    print('Кластер-робастна факторна модель записана у 07_CLUSTER_ROBUST_FACTORIAL.txt')
except Exception as exc:
    (OUT/'07_FACTORIAL_STATUS.txt').write_text('Не обчислено: '+repr(exc),encoding='utf-8')
    print('Факторний аналіз не виконано:',type(exc).__name__,str(exc)[:400])

## 8. MCDM: SRD, DnE, DnM, WSM, TOPSIS

Як у першоджерелі, використовується рівновагова багатокритеріальна схема. Критерії орієнтовані «більше = краще»: ASRI-Audio, clean BA, worst-case BA, mean ROC-AUC та `1−mean EER`. Нормалізація min–max усередині наявних моделей, тому ранги змінюються, якщо додати нову модель. Це **порівняльне**, а не абсолютне свідчення якості.

In [ ]:
criteria=summary.set_index('model')[['ASRI_Audio','clean_BA','worst_scenario_BA','mean_scenario_AUC','mean_scenario_EER']].copy()
criteria['one_minus_EER']=1-criteria.pop('mean_scenario_EER')
assert criteria.notna().all().all(),'MCDM не обчислюємо за відсутніх класів/метрик'
if len(criteria)<2:
    print('MCDM потребує щонайменше двох моделей; пропускаємо')
    mcdm=pd.DataFrame(index=criteria.index)
else:
    spread=criteria.max()-criteria.min()
    normalized=(criteria-criteria.min())/spread.replace(0,np.nan)
    normalized=normalized.fillna(.5)  # однакове значення = не розрізняє моделі
    w=np.ones(len(criteria.columns))/len(criteria.columns)
    ranks=criteria.rank(ascending=False,method='average')
    srd=1-(ranks-1).abs().sum(axis=1)/(len(criteria.columns)*(len(criteria)-1))
    difference=1-normalized
    dne=1-np.sqrt(np.sum(w*(difference**2),axis=1))
    dnm=1-np.sum(w*np.abs(difference),axis=1)
    wsm=(normalized*w).sum(axis=1)
    weighted=normalized*w
    ideal=weighted.max(axis=0);anti=weighted.min(axis=0)
    dpos=np.sqrt(((weighted-ideal)**2).sum(axis=1));dneg=np.sqrt(((weighted-anti)**2).sum(axis=1))
    topsis=dneg/(dpos+dneg).replace(0,np.nan)
    topsis=topsis.fillna(.5)
    mcdm=pd.DataFrame({'SRD_benefit':srd,'DnE_benefit':dne,'DnM_benefit':dnm,'WSM':wsm,'TOPSIS':topsis})
    mcdm['MCDM_consensus']=mcdm.mean(axis=1)
    mcdm=mcdm.sort_values('MCDM_consensus',ascending=False)
    mcdm['rank']=mcdm.MCDM_consensus.rank(ascending=False,method='min').astype(int)
    mcdm.to_csv(OUT/'08_MCDM_RANKING.csv')
    display(mcdm)

## 9. Графіки, приклади аудіо, контроль звіту

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,5))
wide=category.pivot(index='model',columns='family',values='BA').reindex(summary.model)
wide.plot(kind='bar',ax=axes[0],ylim=(0,1),title='Balanced Accuracy за категоріями (validation)',rot=15)
axes[0].set_ylabel('Balanced Accuracy [0–1]');axes[0].grid(axis='y',alpha=.2)
for _,r in ci.iterrows():
    pass
x=np.arange(len(summary))
ci_ord=ci.set_index('model').loc[summary.model]
y=ci_ord.estimate.values
err=np.vstack([y-ci_ord.ci_low.values,ci_ord.ci_high.values-y]);err=np.maximum(err,0)
axes[1].errorbar(x,y,yerr=err,fmt='o',capsize=4)
axes[1].set_xticks(x,summary.model,rotation=15)
axes[1].set_ylim(0,1);axes[1].set_ylabel('ASRI-Audio [0–1]')
axes[1].set_title('ASRI-Audio ± 95% pair-bootstrap CI (НЕ voice-level)')
axes[1].grid(axis='y',alpha=.2)
plt.tight_layout(); fig.savefig(OUT/'09_CATEGORIES_ASRI_CI.png',dpi=160);plt.show()

heat=metrics.pivot(index='scenario',columns='model',values='BA').reindex(SC.scenario)
fig,ax=plt.subplots(figsize=(8+len(models),9));img=ax.imshow(heat.to_numpy(float),vmin=0,vmax=1,aspect='auto')
ax.set_yticks(range(len(heat)),heat.index);ax.set_xticks(range(len(heat.columns)),heat.columns,rotation=25)
ax.set_title('Сценарна матриця Balanced Accuracy (validation)')
fig.colorbar(img,ax=ax,label='Balanced Accuracy');plt.tight_layout()
fig.savefig(OUT/'09_SCENARIO_HEATMAP.png',dpi=160);plt.show()

# Прослуховування тільки з validation (hidden не торкаємося); опційно замінити на train
example=manifest.iloc[0]
wave=read_wave(example.path,getattr(example,'segment_start_sec',0),example.pair_id)
print('Аудіо для ручної перевірки:',example.pair_id, 'label=',int(example.label),'scenario=noise_snr15')
display(Audio(wave,rate=SR));display(Audio(apply_scenario(wave,next(s for s in SCENARIOS if s['scenario']=='noise_snr15'),f'{example.pair_id}/{example.label}'),rate=SR))
fig,ax=plt.subplots(1,1,figsize=(12,3));ax.specgram(wave,NFFT=512,Fs=SR,noverlap=384)
ax.set_title('Спектрограма — validation, clean');ax.set_xlabel('Час (с)');ax.set_ylabel('Частота (Гц)')
plt.tight_layout();fig.savefig(OUT/'09_EXAMPLE_SPECTROGRAM.png',dpi=140);plt.show()

report={'experiment':'scenario_adaptive_EVALUATION_ONLY', 'training_performed':False,
        'article':'https://doi.org/10.3390/make8060161', 'protocol':PROTOCOL_ID,
        'models':models,'number_scenarios':len(SCENARIOS),'evaluation_split':list(EVAL_SPLITS),
        'number_pairs':int(manifest.pair_id.nunique()),'number_prediction_rows':len(predictions),
        'risk':'Only one independently identified fake voice in validation; tests are conditional on that voice, not multi-generator generalization',
        'hidden_test_accessed':False}
(OUT/'10_REVIEW_SUMMARY.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print('Відтворюваний звіт збережено:',OUT)

## 10. Єдиний ZIP для перевірки — без аудіо й ваг моделей

Збираємо тільки протокол, контрольні дані, таблиці й графіки. **Жодних оригінальних WAV, TTS WAV, checkpoint або персональних текстів** у звітному архіві. `01_VALIDATION_MANIFEST.csv` містить серверні шляхи й тексти, тому **не включається**.

In [ ]:
import zipfile
archive=RESULTS_ROOT/f'SAADE_SCENARIO_EVALUATION_v7_{RUN_TAG}_FOR_REVIEW.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for f in sorted(OUT.glob('*')):
        if f.is_file() and f.name not in ('01_VALIDATION_MANIFEST.csv',):
            z.write(f,arcname=f.name)
print('Надіслати мені:',archive)
print('Розмір ZIP:',round(archive.stat().st_size/1024/1024,2),'MB')

## 11. Hidden test — ЗАКРИТО

Ця комірка не відкриває hidden test і не змінює прапорець. Після обговорення validation результатів окремо зафіксуємо моделі, пороги, сценарії, маніфест і виконаємо **один незалежний фінальний запуск**. Не копіюйте `hidden_test` в `EVAL_SPLITS`: він блокується кодом. Параметр `ALLOW_HIDDEN_TEST_ONCE` за замовчуванням `False`, а фактична реалізація винесена в окремий погоджуваний протокол, щоб не відкрити тест випадково.

In [ ]:
if ALLOW_HIDDEN_TEST_ONCE:
    raise RuntimeError('Фінальний hidden test потребує окремого замороженого протоколу (хеші моделей, пороги, сценарії). У v7 він навмисно НЕ виконується.')
print('Hidden test захищений. Жодних hidden прогнозів не створено.')